In [1]:
import torch
import logging
from torch import nn 
from dataclasses import dataclass
import os
import re
import time
import math
import torch
import torch.nn as nn
from torch import optim
import pandas as pd


%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [1]:
from datasets import load_dataset
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")


ImportError: cannot import name 'load_dataset' from 'datasets' (/home/pluto/nn-learning/deep-learning/transformers/baby-transformer/baby_gemma/datasets.py)

In [ ]:
from datasets import TinyShakeSpeareDataset, WikiText103Dataset
from tokenizer import SimpleTokenizer
from data_module import TextDataModule

@dataclass 
class TinyShakeSpeareDatasetCfg:
    seq_len = 64
    batch_size = 1 
    split_ratios = (0.80, 0.10, 0.10)
    pin_memory: bool = True
    num_workers: int = 0
    outfile = 'data/tinyshakespeare.txt'
    dataset = TinyShakeSpeareDataset

@dataclass
class WikiText103DatasetCfg:
    data_dir = data/wikitext-103-raw
    seq_len: int = 64
    batch_size: int = 64
    split_ratios: Tuple[float, float, float] = (0.80, 0.10, 0.10)
    pin_memory: bool = True
    num_workers: int = 4
    data_dir: str = "data/wikitext-103-raw"
    split: str = "train"
    dataset: WikiText103Dataset




cfg = TinyShakeSpeareDatasetCfg()
dataset = cfg.dataset(cfg.outfile)
text = dataset.load()

tokenizer = SimpleTokenizer()
tokenizer.fit(text)
data = tokenizer.encode(text)

dm = TextDataModule(data, cfg)
train_loader= dm.train_dataloader()
val_loader = dm.val_dataloader()
test_laoder = dm.test_dataloader()

seq_len = cfg.seq_len
batch_size = cfg.batch_size

2026-09-21 22:11:28,257 - INFO - Using existing dataset file at data/tinyshakespeare.txt
2026-09-21 22:11:28,261 - INFO - Dataset length: 1115394 characters
2026-09-21 22:11:28,261 - INFO - Sample text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A
2026-09-21 22:11:28,264 - INFO - Constructing tokenizer vocabulary
2026-09-21 22:11:28,291 - INFO - Tokenizer vocabulary constructed with 42197 unique words


In [10]:
import torch
import math
from collections import Counter

# Count unigram frequencies on training set
all_train_tokens = text.split(' ')
all_train_tokens = all_train_tokens[:math.floor(len(all_train_tokens)*.90)]
train_counts = Counter(all_train_tokens)
total_tokens = sum(train_counts.values())
p_train = {k: v / total_tokens for k, v in train_counts.items()}

# Theoretical Unigram Entropy (Zero-context floor)
H_unigram = -sum(p * math.log(p) for p in p_train.values())
print(f"Theoretical Unigram Entropy Floor: {H_unigram:.2f}")

# Cross-entropy of an optimal unigram model evaluated on the Validation set
# (Accounts for out-of-vocabulary and missing mass)
all_val_tokens = all_train_tokens[math.floor(len(all_train_tokens)*.90):]
val_counts = Counter(all_val_tokens)
val_total = sum(val_counts.values())

# Assume small laplace smoothing for unseen tokens
smoothed_prob = 1e-6
val_loss_unigram = -sum(
    (count / val_total) * math.log(p_train.get(tok, smoothed_prob))
    for tok, count in val_counts.items()
)
print(f"Validation Unigram Loss Floor: {val_loss_unigram:.2f}")

Theoretical Unigram Entropy Floor: 7.86
Validation Unigram Loss Floor: 7.79


In [3]:
from transformer import BabyTransformer, Transformer
from layers.norm import RMSNorm
from layers.attention import AttentionHead
from layers.base import FFN

class AbsolutePositonalEncoder(nn.Module):

    def __init__(self, seq_len, d_model):
        super().__init__()
        self.seq_len = seq_len
        self.emb = nn.Embedding(seq_len, d_model)

    def forward(self, x):
        _,seq_len = x.shape
        indices = torch.arange(0, seq_len, device=x.device)
        o = self.emb(indices)
        return o 


# x = torch.tensor(
#     [
#         [
#             [1.0, 2.0, 3.0],
#             [4.0, 5.0, 6.0],
#         ],
#         [
#             [7.0, 8.0, 9.0],
#             [10.0, 11.0, 12.0],            
#         ]
#     ]
# )

# #print(x.shape)
# pos_encoder = AbsolutePositonalEncoder(2, 3)
# o = pos_encoder(x)
# #print(o.shape)

# print(x)
# print(o)
# z = x + o
# print(z)

 

In [4]:
import torch.nn.functional as F
class BabyTransformerPositional(Transformer):
    '''
    Embeddings, no position information, 
    single-head scaled dot-product attention
    '''

    def __init__(self, itos: dict, stoi: dict, seq_len: int, vocab_size: int, d_model: int,
                 dropout: int =0.3):
        super().__init__(itos, stoi, seq_len, vocab_size, d_model)

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = AbsolutePositonalEncoder(seq_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.prenorm_1 = RMSNorm(d_model)
        self.head = AttentionHead(d_model, seq_len, dropout)
        self.attention_dropout = nn.Dropout(dropout)
        self.prenorm_2 = RMSNorm(d_model)
        self.ffn = FFN(d_model, d_model*4, dropout)
        self.ffn_dropout = nn.Dropout(dropout)
        self.prenorm_3 = RMSNorm(d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)


    def forward(self, x):
        emb = self.embedding(x) 
        pos_emb = self.pos_encoder(x)
        emb = self.emb_dropout(emb + pos_emb)
        x_norm = self.prenorm_1(emb)
        head_o = emb + self.attention_dropout(self.head(x_norm))
        head_o_norm = self.prenorm_2(head_o)
        ffn_o = head_o + self.ffn_dropout(self.ffn(head_o_norm))
        ffn_o_norm = self.prenorm_3(ffn_o)
        logits = self.linear(ffn_o_norm) 
        # logits = F.linear(ffn_o_norm, self.embedding.weight)
        return logits



In [5]:

from torch import optim

d_model = 128
epochs = 1
max_seq_len = seq_len
vocab_size = tokenizer.vocab_size

model = BabyTransformerPositional(
    tokenizer.itos,
    tokenizer.stoi,
    max_seq_len, vocab_size, d_model)


def evaluate(model, val_loader, criterion, device, max_batches=None):
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_idx, (batch_x, batch_y) in enumerate(val_loader):
            if max_batches is not None and batch_idx >= max_batches:
                break 

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            B, S, d_out = logits.shape

            all_seqs = logits.view(B*S, d_out)
            y = batch_y.view(B*S)
            loss = criterion(all_seqs, y)
            val_loss += loss.item()

    return val_loss / len(val_loader)



def train_loop(model, train_loader, val_loader=None, num_epochs=5, lr=0.001):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.1,
        patience=3,
    )
    
    print(
        f"Initialized {model.__class__.__name__} | "
        f"vocab_size={model.vocab_size}, d_model={model.d_model}, seq_len={model.seq_len}"
    )
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (batch_x, batch_y) in enumerate(train_loader):

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            B, S, d_out = logits.shape

            all_seqs = logits.view(B*S, d_out)
            y = batch_y.view(B*S)
            loss = criterion(all_seqs, y)

            if batch_idx % 100 == 0:
                logging.info(
                    f"Epoch {epoch+1}/{num_epochs} | "
                    f"Batch {batch_idx}/{len(train_loader)} | "
                    f"Loss: {loss.item():.4f}"
                )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_train_loss = running_loss / len(train_loader)
        history["train_loss"].append(epoch_train_loss)


        # Validation evaluation
        if val_loader is not None:
            epoch_val_loss = evaluate(model, val_loader, criterion, device)
            history["val_loss"].append(epoch_val_loss)
            scheduler.step(epoch_val_loss)
            logging.info(
                f"--> Epoch {epoch+1}/{num_epochs} Complete | "
                f"Train Loss: {epoch_train_loss:.4f} | "
                f"Val Loss: {epoch_val_loss:.4f}"
            )
        else:
            scheduler.step(epoch_train_loss)
            logging.info(
                f"--> Epoch {epoch+1}/{num_epochs} Complete | "
                f"Train Loss: {epoch_train_loss:.4f}"
            )

train_loop(model, train_loader, val_loader, num_epochs=10)

Initialized BabyTransformerPositional | vocab_size=42197, d_model=128, seq_len=64


2026-09-20 11:45:11,259 - INFO - Epoch 1/10 | Batch 0/2123 | Loss: 10.7993
2026-09-20 11:45:11,709 - INFO - Epoch 1/10 | Batch 100/2123 | Loss: 8.9505
2026-09-20 11:45:12,067 - INFO - Epoch 1/10 | Batch 200/2123 | Loss: 9.0632
2026-09-20 11:45:12,411 - INFO - Epoch 1/10 | Batch 300/2123 | Loss: 7.9222
2026-09-20 11:45:12,745 - INFO - Epoch 1/10 | Batch 400/2123 | Loss: 8.4062
2026-09-20 11:45:13,069 - INFO - Epoch 1/10 | Batch 500/2123 | Loss: 8.5851
2026-09-20 11:45:13,395 - INFO - Epoch 1/10 | Batch 600/2123 | Loss: 7.4604
2026-09-20 11:45:13,729 - INFO - Epoch 1/10 | Batch 700/2123 | Loss: 7.9687
2026-09-20 11:45:14,054 - INFO - Epoch 1/10 | Batch 800/2123 | Loss: 9.2497
2026-09-20 11:45:14,379 - INFO - Epoch 1/10 | Batch 900/2123 | Loss: 7.8891
2026-09-20 11:45:14,719 - INFO - Epoch 1/10 | Batch 1000/2123 | Loss: 8.7256
2026-09-20 11:45:15,050 - INFO - Epoch 1/10 | Batch 1100/2123 | Loss: 8.3758
2026-09-20 11:45:15,378 - INFO - Epoch 1/10 | Batch 1200/2123 | Loss: 7.7399
2026-09-20

In [6]:


class BenchmarkHarness:
    def __init__(self, train_loader, val_loader, total_steps=500, lr=1e-3, seed=42):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.total_steps = total_steps
        self.lr = lr
        self.seed = seed
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.results = []

    def _reset_seed(self):
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    def _evaluate(self, model, max_batches=50):
        model.eval()
        total_loss = 0.0
        total_batches = 0
        with torch.no_grad():
            for idx, (batch_x, batch_y) in enumerate(self.val_loader):
                if max_batches and idx >= max_batches:
                    break
                batch_x = batch_x.to(self.device)
                batch_y = batch_y.to(self.device)
                logits = model(batch_x)
                B, S, V = logits.shape
                loss = self.criterion(logits.view(B * S, V), batch_y.view(B * S))
                total_loss += loss.item()
                total_batches += 1
        return total_loss / max(1, total_batches)

    def run_candidate(self, name: str, model_builder_fn, metadata: dict = None):
        self._reset_seed()
        model = model_builder_fn().to(self.device)
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        optimizer = optim.AdamW(model.parameters(), lr=self.lr, weight_decay=0.01)
        data_iter = iter(self.train_loader)

        best_val_loss = float("inf")
        start_time = time.time()

        print(f"--> Benchmarking: {name} ({total_params:,} params)")

        for step in range(1, self.total_steps + 1):
            try:
                batch_x, batch_y = next(data_iter)
            except StopIteration:
                data_iter = iter(self.train_loader)
                batch_x, batch_y = next(data_iter)

            model.train()
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            logits = model(batch_x)
            B, S, V = logits.shape
            loss = self.criterion(logits.view(B * S, V), batch_y.view(B * S))

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if step % 50 == 0 or step == self.total_steps:
                val_loss = self._evaluate(model)
                best_val_loss = min(best_val_loss, val_loss)

        wall_time = time.time() - start_time
        final_val_loss = self._evaluate(model)
        val_ppl = math.exp(final_val_loss) if final_val_loss < 20 else float("inf")

        run_entry = {
            "Model": name,
            "Positional Emb": metadata.get("pos_emb", "None") if metadata else "None",
            "Attention Type": metadata.get("attn_type", "Single-Head") if metadata else "Single-Head",
            "Norm": metadata.get("norm", "None") if metadata else "None",
            "Parameters": f"{total_params:,}",
            "Best Val Loss": round(best_val_loss, 4),
            "Final Val Loss": round(final_val_loss, 4),
            "Final PPL": round(val_ppl, 1),
            "Steps/sec": round(self.total_steps / wall_time, 2),
        }
        self.results.append(run_entry)

    def summary(self, readme_path="../README.md"):
        df = pd.DataFrame(self.results).sort_values(by="Best Val Loss")
        md_table = df.to_markdown(index=False)

        # 1. Print directly to terminal/cell output
        print("\n" + "=" * 35 + " BENCHMARK LEADERBOARD " + "=" * 35)
        print(md_table)
        print("=" * 93 + "\n")

        # 2. Write or update README.md cleanly using markers
        start_marker = "<!-- BENCHMARK_START -->"
        end_marker = "<!-- BENCHMARK_END -->"
        section_content = f"{start_marker}\n## Model Benchmark Results\n\n{md_table}\n{end_marker}"

        if os.path.exists(readme_path):
            with open(readme_path, "r", encoding="utf-8") as f:
                content = f.read()

            if start_marker in content and end_marker in content:
                # Replace existing benchmark table in place
                pattern = rf"{re.escape(start_marker)}.*?{re.escape(end_marker)}"
                updated_content = re.sub(pattern, section_content, content, flags=re.DOTALL)
            else:
                # Append to existing README
                updated_content = content.rstrip() + f"\n\n{section_content}\n"
        else:
            # Create fresh README
            updated_content = f"# Transformer Experiments\n\n{section_content}\n"

        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(updated_content)

        print(f"Benchmark results successfully written to {readme_path}")
        return df

In [7]:
harness = BenchmarkHarness(train_loader, val_loader, total_steps=300)

harness.run_candidate(
    name="Baseline",
    model_builder_fn=lambda: BabyTransformer(tokenizer.itos, tokenizer.stoi, max_seq_len, vocab_size, d_model),
    metadata={"pos_emb": "None", "attn_type": "Single-Head Causal", "norm": "RMSNorm"}
)

# Prints Markdown to stdout and updates README.md
harness.summary()

--> Benchmarking: Baseline (11,026,261 params)

=================================== BENCHMARK LEADERBOARD ===================================
| Model    | Positional Emb   | Attention Type     | Norm    |   Parameters |   Best Val Loss |   Final Val Loss |   Final PPL |   Steps/sec |
|:---------|:-----------------|:-------------------|:--------|-------------:|----------------:|-----------------:|------------:|------------:|
| Baseline | None             | Single-Head Causal | RMSNorm |   11,026,261 |          8.3875 |           8.4113 |      4497.7 |      184.13 |

Benchmark results successfully written to ../README.md


,Model,Positional Emb,Attention Type,Norm,Parameters,Best Val Loss,Final Val Loss,Final PPL,Steps/sec
0,Baseline,None,Single-Head Causal,RMSNorm,"11,026,261",8.3875,8.4113,4497.7,184.13


In [8]:

model.generate(context="the king shall set you free")


" the king shall set you free again not have said you have not for my part, my consent,\nAs and you were alike,\nAnd he does my soul's redemption,\nIs and then their toes\nUnplagued to have learn'd\nTo the king you had rather I am a few to him to make me from the common people.\n\nSecond in a man of"